In [ ]:
# Actually training

In [3]:
import numpy as np

# Load Standard dataset
data_std = np.load("Medium reversi_dataset.npz")
X_nor, y_nor = data_std["X"], data_std["y"]

# Load Hard dataset
data_hard = np.load("Hard reversi_dataset.npz")
X_hard, y_hard = data_hard["X"], data_hard["y"]

print("Standard:", X_nor.shape, y_nor.shape)
print("Hard:", X_hard.shape, y_hard.shape)

Standard: (1830, 8, 8, 1) (1830, 64)
Hard: (3660, 8, 8, 1) (3660, 64)


In [4]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_normal_cnn():
    model = models.Sequential([
        layers.Input(shape=(8, 8, 1)),  # board as 8x8 grid
        layers.Conv2D(32, (3,3), activation='relu', padding='same'),
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dense(64, activation='softmax')  # one output per cell
    ])
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

normal_model = build_normal_cnn()
normal_model.fit(X_nor, y_nor, epochs=20, batch_size=64, validation_split=0.1)
normal_model.save("reversi_normal_cnn.h5")

Epoch 1/20
26/26 ━━━━━━━━━━━━━━━━━━━━ 3s 30ms/step - accuracy: 0.4876 - loss: 2.6120 - val_accuracy: 0.6557 - val_loss: 2.2320
Epoch 2/20
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.9381 - loss: 0.3341 - val_accuracy: 0.7541 - val_loss: 1.8086
Epoch 3/20
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.9787 - loss: 0.0882 - val_accuracy: 0.8852 - val_loss: 1.2007
Epoch 4/20
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.9909 - loss: 0.0332 - val_accuracy: 0.9235 - val_loss: 1.4157
Epoch 5/20
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.9957 - loss: 0.0146 - val_accuracy: 0.9235 - val_loss: 1.4415
Epoch 6/20
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.9982 - loss: 0.0112 - val_accuracy: 0.9454 - val_loss: 1.4481
Epoch 7/20
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.9982 - loss: 0.0073 - val_accuracy: 0.9454 - val_loss: 1.4844
Epoch 8/20
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 1.0000 - loss: 0.0037 - val_accuracy: 0.9454 - v

In [5]:
def build_hard_cnn():
    model = models.Sequential([
        layers.Input(shape=(8, 8, 1)),
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(64, activation='softmax')
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

hard_model = build_hard_cnn()
hard_model.fit(X_hard, y_hard, epochs=50, batch_size=128, validation_split=0.1)
hard_model.save("reversi_hard_cnn.h5")

Epoch 1/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 6s 133ms/step - accuracy: 0.5698 - loss: 2.1972 - val_accuracy: 0.6721 - val_loss: 3.9895
Epoch 2/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 3s 104ms/step - accuracy: 0.9390 - loss: 0.3743 - val_accuracy: 0.8470 - val_loss: 3.8429
Epoch 3/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 3s 125ms/step - accuracy: 0.9821 - loss: 0.1342 - val_accuracy: 0.8661 - val_loss: 3.7229
Epoch 4/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 3s 114ms/step - accuracy: 0.9897 - loss: 0.0736 - val_accuracy: 0.9044 - val_loss: 3.6025
Epoch 5/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 3s 103ms/step - accuracy: 0.9979 - loss: 0.0474 - val_accuracy: 0.9044 - val_loss: 3.4722
Epoch 6/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 3s 119ms/step - accuracy: 0.9976 - loss: 0.0315 - val_accuracy: 0.9044 - val_loss: 3.3294
Epoch 7/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 3s 120ms/step - accuracy: 0.9982 - loss: 0.0235 - val_accuracy: 0.9044 - val_loss: 3.1790
Epoch 8/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 3s 125ms/step - accuracy: 0.9985 - loss: 0.0204 - val_accuracy: 0.

In [ ]:
from tensorflow.keras.models import load_model
normal_model = load_model("reversi_normal_cnn.h5")
hard_model   = load_model("reversi_hard_cnn.h5")

probs = normal_model.predict(board.reshape(1,8,8,1))[0]
for r,c in valid_moves(board, player):
    idx = r*8 + c
    # pick move with highest probability among legal ones


In [ ]:
import numpy as np
normal_model = build_normal_cnn()
normal_model.fit(X_nor, y_nor, epochs=20)

weights_dict = {
    "weights": [w.numpy() for w in normal_model.get_weights()],
    "biases":  []
}

np.save("reversi_model_normal.npy", weights_dict)

hard_model = build_hard_cnn()
hard_model.fit(X_hard, y_hard, epochs=50)

weights_dict = {
    "weights": [w.numpy() for w in hard_model.get_weights()],
    "biases":  []
}

np.save("reversi_model_hard.npy", weights_dict)